WhatNet - EMNIST-mnist

In [ ]:
#  MNIST: 70,000 chars, 10 balanced classes (0-9)
#  7,000 samples/class. Classic benchmark — aim for state-of-the-art accuracy.
#  MNIST is cleaner than EMNIST Digits (only one author source per sample).

In [ ]:
# importing necessary dependencies
import os, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

#  CONFIG  —  MNIST: 70,000 chars, 10 balanced classes (0-9)
DATASET      = "mnist"
NUM_CLASSES  = 10
IMG          = 28
BS           = 56
EPOCHS       = 100
LR           = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.1
WARMUP_EP    = 3
VAL_SPLIT    = 0.10
RESULTS_DIR  = "./results/mnist"
AUTOTUNE     = tf.data.AUTOTUNE

LABELS = [str(d) for d in range(10)]

os.makedirs(RESULTS_DIR, exist_ok=True)

#  DATA LOADING
print(f"[INFO] Loading {DATASET} via tensorflow_datasets ...")
train_raw, info = tfds.load(
    DATASET, split="train", as_supervised=True, with_info=True, shuffle_files=True,
)
test_raw = tfds.load(DATASET, split="test", as_supervised=True)

total   = info.splits["train"].num_examples
n_val   = int(total * VAL_SPLIT)
n_train = total - n_val
print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {info.splits['test'].num_examples:,}")

#  PREPROCESSING
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 127.5 - 1.0
    label = tf.cast(label, tf.int32)
    return image, label

def augment(image, label):
    image = tf.image.random_brightness(image, 0.15)
    image = tf.image.random_contrast(image, 0.85, 1.15)
    image = tf.pad(image, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    image = tf.image.random_crop(image, [IMG, IMG, 1])
    # NO horizontal flip
    image = image + tf.random.normal(tf.shape(image), stddev=0.02)
    image = tf.clip_by_value(image, -1.0, 1.0)
    return image, label

def to_onehot(image, label):
    return image, tf.one_hot(label, NUM_CLASSES)

train_raw = train_raw.shuffle(10_000, seed=SEED, reshuffle_each_iteration=False)
val_raw   = train_raw.take(n_val)
train_raw = train_raw.skip(n_val)

train_ds = (
    train_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
val_ds = (
    val_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
test_ds = test_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BS).prefetch(AUTOTUNE)
test_ds_oh = (
    test_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)

steps_per_epoch = n_train // BS
total_steps     = steps_per_epoch * EPOCHS
print(f"[INFO] Steps/epoch: {steps_per_epoch}")

#  BUILDING BLOCKS
def gelu(x): return tf.nn.gelu(x)

def channel_attention(x, channels, reduction=4):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(max(1, channels // reduction), activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def residual_block(x, channels):
    r = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, r]); x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_ch, out_ch):
    if in_ch != out_ch:
        x = layers.Conv2D(out_ch, 1, use_bias=False)(x)
        x = layers.BatchNormalization()(x); x = layers.Activation(gelu)(x)
    r1  = residual_block(x,  out_ch)
    r2  = residual_block(r1, out_ch)
    r3  = residual_block(r2, out_ch)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_ch, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out); out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_ch, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out); out = layers.Activation(gelu)(out)
    return out

def adaptive_filter_capsule(x, num_classes, capsule_dim=16):
    """
    Lightweight capsule-like routing. Projects the feature vector into a
    (num_classes × capsule_dim) tensor, uses the original feature as a
    per-class filter, and sums to produce class-discriminative scores.
    No dynamic routing — O(n) cost.
    """
    h = layers.Dense(256, activation=gelu)(x)
    h = layers.Dense(num_classes * capsule_dim)(h)
    h = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps = layers.Multiply()([x_sliced, h])
    caps = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps = layers.BatchNormalization()(caps)
    return caps

#  MODEL
def build_model(num_classes=10, image_size=28):
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # Stem
    t = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t = layers.BatchNormalization()(t); t = layers.Activation(gelu)(t)
    h = layers.Conv2D(16, (1, 5), padding="same", use_bias=False)(inp)
    h = layers.BatchNormalization()(h); h = layers.Activation(gelu)(h)
    v = layers.Conv2D(16, (5, 1), padding="same", use_bias=False)(inp)
    v = layers.BatchNormalization()(v); v = layers.Activation(gelu)(v)

    stem = layers.Concatenate()([t, h, v])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem); stem = layers.Activation(gelu)(stem)

    # Encoder
    enc1 = dense_res_block(stem, 64,  64)
    enc2 = dense_res_block(enc1, 64,  128)
    enc3 = dense_res_block(enc2, 128, 256)

    # Multi-scale GAP fusion
    gap1 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc1))
    gap2 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc2))
    gap3 = layers.GlobalAveragePooling2D()(enc3)
    fused_gap = layers.Concatenate(name="multiscale_fused")([gap1, gap2, gap3])  # (B, 512)

    # Adaptive Filter Capsule (AFC)
    afc_out = adaptive_filter_capsule(fused_gap, K)   # (B, K)

    # Dense classification head (residual path alongside AFC)
    x = layers.Dense(128, use_bias=False, name="head_dense")(fused_gap)
    x = layers.LayerNormalization(name="head_ln")(x)
    x = layers.Activation(gelu, name="head_act")(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(K, name="head_logits")(x)        # (B, K)

    # Gated fusion: AFC scores + dense-head logits
    combined   = layers.Concatenate(name="gate_input")([x, afc_out])
    gate       = layers.Dense(2, activation="softmax", name="gate")(combined)  # (B, 2)
    x_scaled   = layers.Lambda(
        lambda t: t[0] * t[1][:, 0:1], name="gate_dense")([x, gate])
    afc_scaled = layers.Lambda(
        lambda t: t[0] * t[1][:, 1:2], name="gate_afc")([afc_out, gate])
    out = layers.Add(name="logits")([x_scaled, afc_scaled])

    return Model(inputs=inp, outputs=out, name="WhatNet_MNIST")

#  LR SCHEDULE
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base, total_steps, warmup_steps):
        self.base         = base
        self.total_steps  = tf.cast(total_steps,  tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup_lr = self.base * step / tf.maximum(self.warmup_steps, 1.0)
        progress  = (step - self.warmup_steps) / tf.maximum(
            self.total_steps - self.warmup_steps, 1.0)
        cosine_lr = tf.maximum(self.base * 0.5 * (1.0 + tf.cos(np.pi * progress)), 1e-6)
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)
    def get_config(self):
        return {"base": self.base, "total_steps": int(self.total_steps),
                "warmup_steps": int(self.warmup_steps)}

#  TRAIN
model        = build_model(NUM_CLASSES, IMG)
warmup_steps = steps_per_epoch * WARMUP_EP
lr_sch       = WarmupCosineDecay(LR, total_steps, warmup_steps)

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=lr_sch, weight_decay=WEIGHT_DECAY),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True, label_smoothing=LABEL_SMOOTH),
    metrics=["accuracy"],
    jit_compile=True,
)
print(f"[INFO] Params: {model.count_params():,}")

history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=[
        ModelCheckpoint(os.path.join(RESULTS_DIR, "best_model.keras"),
                        monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=10,
                      restore_best_weights=True, verbose=1),
    ],
)

#  EVALUATE
test_loss, test_acc = model.evaluate(test_ds_oh, verbose=0)
test_acc *= 100.0

tp = np.zeros(NUM_CLASSES); fp = np.zeros(NUM_CLASSES); fn = np.zeros(NUM_CLASSES)
correct_per_class = np.zeros(NUM_CLASSES); total_per_class = np.zeros(NUM_CLASSES)

for images, labels in test_ds:
    preds = tf.argmax(model(images, training=False), axis=1).numpy()
    lbls  = labels.numpy()
    for c in range(NUM_CLASSES):
        tp[c] += np.sum((preds == c) & (lbls == c))
        fp[c] += np.sum((preds == c) & (lbls != c))
        fn[c] += np.sum((preds != c) & (lbls == c))
        correct_per_class[c] += np.sum(preds[lbls == c] == c)
        total_per_class[c]   += np.sum(lbls == c)

prec          = tp / (tp + fp + 1e-8)
rec           = tp / (tp + fn + 1e-8)
f1            = 2 * prec * rec / (prec + rec + 1e-8)
macro_f1      = float(f1.mean() * 100.0)
per_class_acc = correct_per_class / np.maximum(total_per_class, 1) * 100.0
worst5_idx    = np.argsort(per_class_acc)[:5]

print(f"\n[RESULT] Test Acc : {test_acc:.2f}%")
print(f"[RESULT] Macro F1 : {macro_f1:.2f}%")
print(f"[RESULT] Test Loss: {test_loss:.4f}")
print(f"[RESULT] Params   : {model.count_params():,}")
print(f"\n[RESULT] Worst-5 classes:")
for idx in worst5_idx:
    print(f"         '{LABELS[idx]}' (cls {idx:>2}) = {per_class_acc[idx]:.1f}%")

results = {
    "dataset": "MNIST", "num_classes": NUM_CLASSES,
    "test_acc": round(test_acc, 2), "macro_f1": round(macro_f1, 2),
    "test_loss": round(float(test_loss), 4), "params": model.count_params(),
    "per_class_acc": {LABELS[c]: round(per_class_acc[c], 2) for c in range(NUM_CLASSES)},
}
with open(os.path.join(RESULTS_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
with open(os.path.join(RESULTS_DIR, "history.json"), "w") as f:
    json.dump({k: [float(v) for v in vs] for k, vs in history.history.items()}, f, indent=2)
print(f"\n[INFO] Saved to {RESULTS_DIR}/"); print("[DONE]")

In [ ]:
 [INFO] Loading mnist via tensorflow_datasets ...
[INFO] Train: 54,000 | Val: 6,000 | Test: 10,000
[INFO] Steps/epoch: 964
[INFO] Params: 5,321,420
Epoch 1/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 146s 104ms/step - accuracy: 0.3881 - loss: 1.9540 - val_accuracy: 0.9650 - val_loss: 0.6701
Epoch 2/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 67s 70ms/step - accuracy: 0.9650 - loss: 0.7146 - val_accuracy: 0.9857 - val_loss: 0.5570
Epoch 3/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 67s 69ms/step - accuracy: 0.9841 - loss: 0.6086 - val_accuracy: 0.9758 - val_loss: 0.5855
Epoch 4/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 69ms/step - accuracy: 0.9862 - loss: 0.5673 - val_accuracy: 0.9857 - val_loss: 0.5418
Epoch 5/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 69ms/step - accuracy: 0.9889 - loss: 0.5460 - val_accuracy: 0.9837 - val_loss: 0.5508
Epoch 6/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 67s 69ms/step - accuracy: 0.9896 - loss: 0.5364 - val_accuracy: 0.9870 - val_loss: 0.5388
Epoch 7/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 67s 69ms/step - accuracy: 0.9907 - loss: 0.5315 - val_accuracy: 0.9910 - val_loss: 0.5303
Epoch 8/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 69ms/step - accuracy: 0.9922 - loss: 0.5264 - val_accuracy: 0.9868 - val_loss: 0.5399
Epoch 9/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 68ms/step - accuracy: 0.9929 - loss: 0.5250 - val_accuracy: 0.9867 - val_loss: 0.5408
Epoch 10/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 68ms/step - accuracy: 0.9931 - loss: 0.5233 - val_accuracy: 0.9895 - val_loss: 0.5308
Epoch 11/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 67s 69ms/step - accuracy: 0.9940 - loss: 0.5224 - val_accuracy: 0.9917 - val_loss: 0.5259
Epoch 12/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 68ms/step - accuracy: 0.9934 - loss: 0.5212 - val_accuracy: 0.9897 - val_loss: 0.5330
Epoch 13/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 67s 69ms/step - accuracy: 0.9948 - loss: 0.5168 - val_accuracy: 0.9923 - val_loss: 0.5242
Epoch 14/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 68ms/step - accuracy: 0.9951 - loss: 0.5171 - val_accuracy: 0.9910 - val_loss: 0.5242
Epoch 15/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 68ms/step - accuracy: 0.9955 - loss: 0.5150 - val_accuracy: 0.9840 - val_loss: 0.5452
Epoch 16/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 69ms/step - accuracy: 0.9954 - loss: 0.5154 - val_accuracy: 0.9700 - val_loss: 0.5787
Epoch 17/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 69ms/step - accuracy: 0.9958 - loss: 0.5135 - val_accuracy: 0.9913 - val_loss: 0.5259
Epoch 18/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 67s 69ms/step - accuracy: 0.9967 - loss: 0.5115 - val_accuracy: 0.9930 - val_loss: 0.5207
Epoch 19/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 67s 69ms/step - accuracy: 0.9967 - loss: 0.5115 - val_accuracy: 0.9938 - val_loss: 0.5198
Epoch 20/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 68ms/step - accuracy: 0.9975 - loss: 0.5097 - val_accuracy: 0.9627 - val_loss: 0.5959
Epoch 21/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 68ms/step - accuracy: 0.9968 - loss: 0.5105 - val_accuracy: 0.9912 - val_loss: 0.5226
Epoch 22/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 68ms/step - accuracy: 0.9976 - loss: 0.5091 - val_accuracy: 0.9935 - val_loss: 0.5195
Epoch 23/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 69ms/step - accuracy: 0.9983 - loss: 0.5071 - val_accuracy: 0.9925 - val_loss: 0.5199
Epoch 24/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 69ms/step - accuracy: 0.9981 - loss: 0.5074 - val_accuracy: 0.9917 - val_loss: 0.5261
Epoch 25/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 68ms/step - accuracy: 0.9985 - loss: 0.5068 - val_accuracy: 0.9930 - val_loss: 0.5209
Epoch 26/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 68ms/step - accuracy: 0.9978 - loss: 0.5080 - val_accuracy: 0.9892 - val_loss: 0.5337
Epoch 27/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 67s 69ms/step - accuracy: 0.9971 - loss: 0.5089 - val_accuracy: 0.9945 - val_loss: 0.5153
Epoch 28/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 67s 69ms/step - accuracy: 0.9991 - loss: 0.5048 - val_accuracy: 0.9947 - val_loss: 0.5144
Epoch 29/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 69ms/step - accuracy: 0.9979 - loss: 0.5078 - val_accuracy: 0.9882 - val_loss: 0.5341
Epoch 30/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 69ms/step - accuracy: 0.9984 - loss: 0.5065 - val_accuracy: 0.9933 - val_loss: 0.5181
Epoch 31/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 69ms/step - accuracy: 0.9981 - loss: 0.5070 - val_accuracy: 0.9942 - val_loss: 0.5167
Epoch 32/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 68ms/step - accuracy: 0.9989 - loss: 0.5050 - val_accuracy: 0.9940 - val_loss: 0.5186
Epoch 33/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 68ms/step - accuracy: 0.9985 - loss: 0.5060 - val_accuracy: 0.9917 - val_loss: 0.5243
Epoch 34/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 68ms/step - accuracy: 0.9987 - loss: 0.5053 - val_accuracy: 0.9940 - val_loss: 0.5165
Epoch 35/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 67s 69ms/step - accuracy: 0.9991 - loss: 0.5044 - val_accuracy: 0.9948 - val_loss: 0.5151
Epoch 36/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 67s 69ms/step - accuracy: 0.9989 - loss: 0.5042 - val_accuracy: 0.9953 - val_loss: 0.5153
Epoch 37/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 67s 69ms/step - accuracy: 0.9992 - loss: 0.5038 - val_accuracy: 0.9960 - val_loss: 0.5130
Epoch 38/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 69ms/step - accuracy: 0.9994 - loss: 0.5034 - val_accuracy: 0.9940 - val_loss: 0.5166
Epoch 39/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 68ms/step - accuracy: 0.9991 - loss: 0.5043 - val_accuracy: 0.9942 - val_loss: 0.5167
Epoch 40/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 68ms/step - accuracy: 0.9991 - loss: 0.5041 - val_accuracy: 0.9920 - val_loss: 0.5235
Epoch 41/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 68ms/step - accuracy: 0.9992 - loss: 0.5039 - val_accuracy: 0.9945 - val_loss: 0.5151
Epoch 42/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 68ms/step - accuracy: 0.9995 - loss: 0.5029 - val_accuracy: 0.9937 - val_loss: 0.5196
Epoch 43/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 69ms/step - accuracy: 0.9988 - loss: 0.5045 - val_accuracy: 0.9943 - val_loss: 0.5167
Epoch 44/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 67s 69ms/step - accuracy: 0.9994 - loss: 0.5032 - val_accuracy: 0.9940 - val_loss: 0.5174
Epoch 45/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 68ms/step - accuracy: 0.9991 - loss: 0.5036 - val_accuracy: 0.9937 - val_loss: 0.5186
Epoch 46/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 68ms/step - accuracy: 0.9997 - loss: 0.5025 - val_accuracy: 0.9947 - val_loss: 0.5146
Epoch 47/100
965/965 ━━━━━━━━━━━━━━━━━━━━ 66s 68ms/step - accuracy: 0.9995 - loss: 0.5027 - val_accuracy: 0.9952 - val_loss: 0.5126
Epoch 47: early stopping
Restoring model weights from the end of the best epoch: 37.

[RESULT] Test Acc : 99.64%
[RESULT] Macro F1 : 99.64%
[RESULT] Test Loss: 0.5113
[RESULT] Params   : 5,321,420

[RESULT] Worst-5 classes:
         '9' (cls  9) = 98.9%
         '7' (cls  7) = 99.5%
         '5' (cls  5) = 99.6%
         '6' (cls  6) = 99.6%
         '0' (cls  0) = 99.6%